# Used Bikes EDA, Statistical Analysis, and Regression-Ready Dataset (Q7–Q32)

In [ ]:

import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from scipy import stats
from scipy.stats import ttest_ind, f_oneway
from sklearn.preprocessing import OneHotEncoder

df = pd.read_csv("used-bikes (4).csv")

# Feature engineering
df["depreciation_pct"] = (df["Original Price"] - df["price"]) / df["Original Price"]
df["price_ratio"] = df["price"] / df["Original Price"]

premium_brands = ["Harley-Davidson","Ducati","Triumph","Indian","MV","BMW"]
df["is_premium"] = df["brand"].isin(premium_brands).astype(int)

owner_map = {
    "First Owner":1,
    "Second Owner":2,
    "Third Owner":3,
    "Fourth Owner Or More":4
}
df["owner_ordinal"] = df["owner"].map(owner_map)

print(df.head())


In [ ]:

# Q7: Price distribution and log transformation
print("Price skewness:", df["price"].skew())

plt.figure(figsize=(6,4))
df["price"].hist(bins=40)
plt.title("Price Distribution")
plt.show()

plt.figure(figsize=(6,4))
np.log1p(df["price"]).hist(bins=40)
plt.title("Log(Price) Distribution")
plt.show()

print("Recommendation: If skewness > 1, use log(price) for regression.")


In [ ]:

# Q8 Brand vs price
brand_price = df.groupby("brand")["price"].mean().sort_values()
print("Lowest average resale value brands")
print(brand_price.head())

print("Highest average resale value brands")
print(brand_price.tail())


In [ ]:

# Q9 kms vs price
corr = df["kms_driven"].corr(df["price"])
print("Correlation:", corr)

plt.figure(figsize=(6,4))
plt.scatter(df["kms_driven"], df["price"], alpha=0.3)
plt.xlabel("kms_driven")
plt.ylabel("price")
plt.title("kms_driven vs price")
plt.show()


In [ ]:

# Q10 age impact
print("Correlation age-price:", df["age"].corr(df["price"]))

sns.regplot(data=df,x="age",y="price",scatter_kws={"alpha":0.2})
plt.show()


In [ ]:

# Q11 depreciation distribution
df["depreciation_pct"].hist(bins=40)
plt.title("Depreciation % Distribution")
plt.show()

print(df["depreciation_pct"].describe())


In [ ]:

# Q12 power vs price
print("Correlation power-price:", df["power"].corr(df["price"]))

plt.scatter(df["power"], df["price"], alpha=0.3)
plt.xlabel("power")
plt.ylabel("price")
plt.show()

# potential outliers
outliers = df[(df["power"]>df["power"].quantile(0.95)) &
              (df["price"]<df["price"].median())]
print(outliers.head())


In [ ]:

# Q13 city effect
metros = ["Delhi","Mumbai","Bangalore"]
city_price = df.groupby("city")["price"].mean().sort_values(ascending=False)
print(city_price.head(15))


In [ ]:

# Q14 visualizations
for col in ["price","kms_driven","age","power"]:
    plt.figure(figsize=(6,4))
    sns.histplot(df[col], kde=True)
    plt.title(col)
    plt.show()

for col in ["price","kms_driven","age","power"]:
    plt.figure(figsize=(6,4))
    sns.boxplot(x=df[col])
    plt.title(col)
    plt.show()


In [ ]:

# Q15 bar charts
df.groupby("brand")["price"].mean().sort_values().plot(kind="bar", figsize=(12,5))
plt.title("Average Price by Brand")
plt.show()

df.groupby("city")["depreciation_pct"].mean().sort_values().tail(20).plot(kind="bar", figsize=(12,5))
plt.title("Average Depreciation by City")
plt.show()


In [ ]:

# Q16 correlation heatmap
cols = ["price","age","kms_driven","power","depreciation_pct"]
sns.heatmap(df[cols].corr(), annot=True, cmap="coolwarm")
plt.show()


In [ ]:

# Q17 summary statistics
summary = pd.DataFrame({
    "mean": df[cols].mean(),
    "median": df[cols].median(),
    "mode": [df[c].mode()[0] for c in cols],
    "std": df[cols].std(),
    "skew": df[cols].skew(),
    "kurtosis": df[cols].kurt()
})
print(summary)


In [ ]:

# Q19 price_ratio analysis
print(df.groupby("brand")["price_ratio"].mean().sort_values())

sns.boxplot(data=df,x="age",y="price_ratio")
plt.show()


In [ ]:

# Q20 owner ordinal
print("Owner is ordinal and can be encoded numerically for regression.")
print(df["owner"].value_counts())


In [ ]:

# Q21 multicollinearity check
print(df[["age","kms_driven","power"]].corr())


In [ ]:

# Q22 premium flag
print(df["is_premium"].value_counts())


In [ ]:

# Q23 outlier detection using IQR
for c in ["price","kms_driven","power"]:
    q1 = df[c].quantile(0.25)
    q3 = df[c].quantile(0.75)
    iqr = q3-q1
    lower = q1 - 1.5*iqr
    upper = q3 + 1.5*iqr
    out = df[(df[c]<lower)|(df[c]>upper)]
    print(c, len(out))


In [ ]:

# Q24 suggested regression features
features = [
    "brand","city","age","kms_driven","power",
    "owner_ordinal","depreciation_pct",
    "price_ratio","is_premium"
]
print(features)


In [ ]:

# Q25 regression-ready dataset
reg_df = pd.get_dummies(
    df[["price","brand","city","age","kms_driven","power",
        "owner_ordinal","depreciation_pct","price_ratio","is_premium"]],
    columns=["brand","city"],
    drop_first=True
)

print(reg_df.shape)
reg_df.head()


In [ ]:

# Q26 metro vs non-metro
df["metro"] = df["city"].isin(["Delhi","Mumbai","Bangalore"])

metro = df[df["metro"]]["price"]
nonmetro = df[~df["metro"]]["price"]

print(ttest_ind(metro, nonmetro, equal_var=False))


In [ ]:

# Q27 ANOVA depreciation across brands
groups = [g["depreciation_pct"].values for _,g in df.groupby("brand")]
print(f_oneway(*groups))


In [ ]:

# Q28 ownership effect on price
groups = [g["price"].values for _,g in df.groupby("owner")]
print(f_oneway(*groups))


In [ ]:

# Q29 age decreases price hypothesis

# H0: Age has no effect on price
# H1: Price decreases as age increases

result = stats.linregress(df["age"], df["price"])
print(result)


In [ ]:

# Q30 power increases price
result = stats.linregress(df["power"], df["price"])
print(result)


In [ ]:

# Q31 High kms vs low kms
median_km = df["kms_driven"].median()

low = df[df["kms_driven"]<=median_km]["price"]
high = df[df["kms_driven"]>median_km]["price"]

print(ttest_ind(low, high, equal_var=False))


In [ ]:

# Q32 Brand and power interaction

import statsmodels.api as sm
from statsmodels.formula.api import ols

model = ols('price ~ C(brand) + power + C(brand):power', data=df).fit()
anova_table = sm.stats.anova_lm(model, typ=2)

print(anova_table)
